# 03 Neural stream inventory and decoder manifest

This notebook bridges the trial catalog and the first decoding notebook.

## Purpose
1. Open every NWB file listed in the session index.
2. Inventory acquisition and processing objects across sessions.
3. Identify candidate neural streams for decoding.
4. Build a decoder manifest that maps each session to a likely neural source.
5. Save paper-style figures, tables, and metadata for the baseline decoder notebook.

> **Reference:** Issar et al. (2020) *J Neurophysiol* 123:1472–1485.  
> All figures in this notebook follow the aesthetic conventions of the reference paper:
> overlaid waveform traces, scatter-over-time with regression, dual-axis line plots,
> session-distribution histograms, and shaded mean ± SE curves.

## Inputs
- `/kaggle/working/tables_session_index/session_master_index.csv`
- `/kaggle/working/tables_session_index/decoder_trial_table.csv`
- Raw NWB files under: `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N`

## Outputs
- `tables_stream_inventory/all_object_manifest.csv`
- `tables_stream_inventory/candidate_stream_manifest.csv`
- `tables_stream_inventory/session_decoder_stream_map.csv`
- `meta_stream_inventory/stream_inventory_report.json`
- `figures_stream_inventory/*.png / *.pdf`

In [ ]:
!pip install pynwb h5py --quiet

In [ ]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings("ignore")
from pynwb import NWBHDF5IO

In [ ]:
# ─────────────────────────────────────────────
# PAPER-STYLE GLOBAL SETTINGS
# Mirrors Issar et al. 2020 (J Neurophysiol)
# ─────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.edgecolor": "0.4",
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

# Paper palette (from Issar et al. Fig. 4–6)
C_BLACK   = "#1a1a1a"
C_MAROON  = "#8B2222"   # % waveforms removed line
C_BLUE    = "#2166AC"   # threshold crossings reference
C_GREEN   = "#1B7837"   # maximum decoding
C_GRAY    = "#888888"   # chance / control
C_ORANGE  = "#E08214"   # SpikingBandPower stream
C_RED_FILL = "#D6604D"  # NAS-helps shading

MARKER_PE  = "o"        # monkey Pe / primary subject
MARKER_WA  = "s"        # monkey Wa / secondary subject

def paper_axes(ax, top=True, right=True):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.85)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=top, right=right)

np.random.seed(42)

## Paths and notebook dependencies

This notebook expects the outputs from `02_session_index_and_trial_catalog.ipynb` to already exist.

In [ ]:
DATASET_DIR = Path("/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N")

IN_TABLE_DIR = Path("/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index")
SESSION_MASTER_CSV    = IN_TABLE_DIR / "session_master_index.csv"
DECODER_TRIAL_TABLE_CSV = IN_TABLE_DIR / "decoder_trial_table.csv"

OUT_FIG_DIR   = Path("/kaggle/working/figures_stream_inventory")
OUT_TABLE_DIR = Path("/kaggle/working/tables_stream_inventory")
OUT_META_DIR  = Path("/kaggle/working/meta_stream_inventory")

OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUT_META_DIR.mkdir(parents=True, exist_ok=True)

# Sample NWB for deep-dive waveform figures
SAMPLE_NWB = DATASET_DIR / "sub-Monkey-N_ses-20200127_ecephys.nwb"

print("DATASET_DIR         :", DATASET_DIR)
print("SESSION_MASTER_CSV  :", SESSION_MASTER_CSV)
print("DECODER_TRIAL_TABLE :", DECODER_TRIAL_TABLE_CSV)
print("SAMPLE_NWB          :", SAMPLE_NWB)

In [ ]:
assert DATASET_DIR.exists(),        f"Missing dataset directory: {DATASET_DIR}"
assert SESSION_MASTER_CSV.exists(), f"Missing prerequisite: {SESSION_MASTER_CSV}"
assert DECODER_TRIAL_TABLE_CSV.exists(), f"Missing prerequisite: {DECODER_TRIAL_TABLE_CSV}"
assert SAMPLE_NWB.exists(),         f"Missing sample NWB: {SAMPLE_NWB}"

session_master_df   = pd.read_csv(SESSION_MASTER_CSV)
decoder_trial_table = pd.read_csv(DECODER_TRIAL_TABLE_CSV)

if "session_date" in session_master_df.columns:
    session_master_df["session_date"] = pd.to_datetime(session_master_df["session_date"], errors="coerce")

if "days_since_first_session" not in session_master_df.columns:
    d0 = session_master_df["session_date"].min()
    session_master_df["days_since_first_session"] = (session_master_df["session_date"] - d0).dt.days

print("session_master_df shape :", session_master_df.shape)
print("decoder_trial_table shape:", decoder_trial_table.shape)
display(session_master_df.head(5))

## Helper functions

The following helpers safely inspect NWB objects without forcing large data reads into memory.

In [ ]:
def safe_attr(obj, attr, default=None):
    try:
        return getattr(obj, attr)
    except Exception:
        return default

def safe_len(x, default=np.nan):
    try:
        return len(x)
    except Exception:
        return default

def safe_shape(x):
    try:
        if hasattr(x, "shape") and x.shape is not None:
            return tuple(x.shape)
        return (len(x),)
    except Exception:
        return None

def to_jsonable(x):
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, list):
        return [to_jsonable(v) for v in x]
    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}
    return str(x)

In [ ]:
def object_path(obj):
    parts = []
    seen  = set()
    cur   = obj
    while cur is not None and id(cur) not in seen:
        seen.add(id(cur))
        name = safe_attr(cur, "name", None)
        if name not in [None, "root"]:
            parts.append(str(name))
        cur = safe_attr(cur, "parent", None)
    return "/".join(reversed(parts))

def object_type(obj):
    ndt = safe_attr(obj, "neurodata_type", None)
    if ndt is not None:
        return str(ndt)
    return obj.__class__.__name__

def infer_container_group(path_text):
    txt = (path_text or "").lower()
    if "acquisition"  in txt: return "acquisition"
    if "processing"   in txt: return "processing"
    if "analysis"     in txt: return "analysis"
    if "stimulus"     in txt: return "stimulus"
    if "intervals"    in txt: return "intervals"
    return "other"

In [ ]:
def describe_nwb_object(obj):
    desc = {
        "name"         : safe_attr(obj, "name",         None),
        "neurodata_type": object_type(obj),
        "path"         : object_path(obj),
        "unit"         : safe_attr(obj, "unit",          None),
        "rate"         : safe_attr(obj, "rate",          None),
        "starting_time": safe_attr(obj, "starting_time", None),
        "description"  : safe_attr(obj, "description",  None),
        "comments"     : safe_attr(obj, "comments",      None),
    }
    data       = safe_attr(obj, "data",       None)
    timestamps = safe_attr(obj, "timestamps", None)
    desc["has_data"]        = data is not None
    desc["has_timestamps"]  = timestamps is not None
    desc["data_shape"]      = safe_shape(data)
    desc["timestamps_shape"]= safe_shape(timestamps)
    desc["container_group"] = infer_container_group(desc["path"])
    return desc

In [ ]:
def candidate_score(row):
    text = " ".join([
        str(row.get("name",         "")),
        str(row.get("neurodata_type","")),
        str(row.get("path",         "")),
        str(row.get("description",  "")),
        str(row.get("comments",     "")),
    ]).lower()

    score = 0
    positive_keywords = {
        "spike": 4, "waveform": 4, "threshold": 4,
        "electrical": 3, "ecephys": 3, "lfp": 3, "mua": 3, "multiunit": 3,
        "event": 2, "timeseries": 1,
    }
    negative_keywords = {
        "eye": -4, "gaze": -4, "pupil": -4,
        "fix": -3, "target": -3, "stim": -3, "reward": -3,
        "position": -2, "cursor": -2, "behavior": -2,
    }
    for k, v in positive_keywords.items():
        if k in text: score += v
    for k, v in negative_keywords.items():
        if k in text: score += v
    if row.get("has_data", False):          score += 1
    if pd.notna(row.get("rate", np.nan)):   score += 1
    if row.get("container_group") in ["acquisition", "processing"]: score += 1
    return score

In [ ]:
def inspect_single_nwb(nwb_path):
    rows = []
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        base_meta = {
            "file_name"          : nwb_path.name,
            "file_path"          : str(nwb_path),
            "identifier"         : safe_attr(nwb, "identifier", None),
            "session_description": safe_attr(nwb, "session_description", None),
            "session_start_time" : str(safe_attr(nwb, "session_start_time", None)),
            "subject_id"         : safe_attr(safe_attr(nwb, "subject", None), "subject_id", None),
        }
        for obj in nwb.all_children():
            try:
                desc = describe_nwb_object(obj)
                if desc["name"] is None:
                    continue
                row = {**base_meta, **desc}
                row["candidate_score"] = candidate_score(row)
                rows.append(row)
            except Exception:
                continue
    return rows

## Build the all-object manifest

We scan every NWB file and create one row per discovered NWB object.
This mirrors the data inventory step described in Issar et al. (2020) §Materials and Methods,
where threshold-crossing streams and spiking-band power streams are identified as the
primary neural sources for decoding.

In [ ]:
nwb_files = sorted(DATASET_DIR.glob("*_ecephys.nwb"))
assert len(nwb_files) > 0, "No NWB files found."

all_rows = []
for i, nwb_path in enumerate(nwb_files, start=1):
    if i % 25 == 0 or i == 1 or i == len(nwb_files):
        print(f"Inspecting {i}/{len(nwb_files)}: {nwb_path.name}")
    rows = inspect_single_nwb(nwb_path)
    all_rows.extend(rows)

all_object_manifest = pd.DataFrame(all_rows)

# Merge session-level fields
merge_cols = [c for c in ["file_name", "session_index", "session_date",
                           "target_style_inferred", "days_since_first_session"]
              if c in session_master_df.columns]
if merge_cols:
    all_object_manifest = all_object_manifest.merge(
        session_master_df[merge_cols].drop_duplicates("file_name"),
        on="file_name", how="left"
    )

all_object_manifest["candidate_flag"] = all_object_manifest["candidate_score"] >= 4
all_object_manifest["data_shape"]      = all_object_manifest["data_shape"].apply(to_jsonable)
all_object_manifest["timestamps_shape"]= all_object_manifest["timestamps_shape"].apply(to_jsonable)

all_object_manifest.to_csv(OUT_TABLE_DIR / "all_object_manifest.csv", index=False)
print("all_object_manifest shape:", all_object_manifest.shape)
display(all_object_manifest.head(10))

In [ ]:
manifest_summary = (
    all_object_manifest
    .groupby(["neurodata_type", "container_group"], dropna=False)
    .size()
    .rename("n_objects")
    .reset_index()
    .sort_values("n_objects", ascending=False)
)
manifest_summary.to_csv(OUT_TABLE_DIR / "object_type_summary.csv", index=False)
display(manifest_summary.head(25))

In [ ]:
candidate_stream_manifest = (
    all_object_manifest
    .loc[all_object_manifest["candidate_flag"]]
    .sort_values(["file_name", "candidate_score"], ascending=[True, False])
    .reset_index(drop=True)
)
candidate_stream_manifest.to_csv(OUT_TABLE_DIR / "candidate_stream_manifest.csv", index=False)
print("candidate_stream_manifest shape:", candidate_stream_manifest.shape)
display(candidate_stream_manifest.head(20))